# Домашнее задание 3. Парсинг, Git и тестирование на Python

**Цели задания:**

* Освоить базовые подходы к web-scraping с библиотеками `requests` и `BeautisulSoup`: навигация по страницам, извлечение HTML-элементов, парсинг.
* Научиться автоматизировать задачи с использованием библиотеки `schedule`.
* Попрактиковаться в использовании Git и оформлении проектов на GitHub.
* Написать и запустить простые юнит-тесты с использованием `pytest`.


В этом домашнем задании вы разработаете систему для автоматического сбора данных о книгах с сайта [Books to Scrape](http://books.toscrape.com). Нужно реализовать функции для парсинга всех страниц сайта, извлечения информации о книгах, автоматического ежедневного запуска задачи и сохранения результата.

Важной частью задания станет оформление проекта: вы создадите репозиторий на GitHub, оформите `README.md`, добавите артефакты (код, данные, отчеты) и напишете базовые тесты на `pytest`.



In [ ]:
! pip install -q schedule pytest # установка библиотек, если ещё не

In [1]:
# Библиотеки, которые могут вам понадобиться
# При необходимости расширяйте список
import time
import re
import json
import logging
from concurrent.futures import ThreadPoolExecutor, wait
from queue import Queue, Empty
import requests
import schedule
from bs4 import BeautifulSoup


## Задание 1. Сбор данных об одной книге (20 баллов)

В этом задании мы начнем подготовку скрипта для парсинга информации о книгах со страниц каталога сайта [Books to Scrape](https://books.toscrape.com/).

Для начала реализуйте функцию `get_book_data`, которая будет получать данные о книге с одной страницы (например, с [этой](http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html)). Соберите всю информацию, включая название, цену, рейтинг, количество в наличии, описание и дополнительные характеристики из таблицы Product Information. Результат достаточно вернуть в виде словаря.

**Не забывайте про соблюдение PEP-8** — помимо качественно написанного кода важно также документировать функции по стандарту:
* кратко описать, что она делает и для чего нужна;
* какие входные аргументы принимает, какого они типа и что означают по смыслу;
* аналогично описать возвращаемые значения.

*P. S. Состав, количество аргументов функции и тип возвращаемого значения можете менять как вам удобно. То, что написано ниже в шаблоне — лишь пример.*

In [2]:
def get_book_data(book_url: str) -> dict:
    """
    Посылает GET запрос по URL адресу страницы с книгой.
    Возвращает информацию о странице с типом объекта BeautifulSoup

    Args:
        book_url (str): URL адрес страницы с книгой
    Returns:
        book (dict): Информация о книге: название, цена, рейтинг,
        количество в наличии, описание и дополнительные характеристики
    """

    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    book = {
        'title': '',
        'price': 0,
        'rating': 0,
        'amount': 0,
        'description': '',
        'additional_info': {}
    }
    logging.debug(f"Fetching book page: {book_url}")
    soup = BeautifulSoup(requests.get(book_url, timeout=10).content, 'html.parser')

    # Название
    if soup.find('h1'):
        book['title'] = soup.find('h1').text
        logging.debug(f"Parsed title: {book['title']}")
    else:
        logging.debug("Title not found on page.")

    # Цена
    if soup.find('p', class_='price_color'):
        book['price'] = soup.find('p', class_='price_color').text
        logging.debug(f"Parsed price: {book['price']}")
    else:
        logging.debug("Price not found on page.")

    # Количество
    if soup.find('p', class_='instock availability'):
        # Извлекаем количество в числовом формате из строки "In stock ({amount} available)"
        amount = re.findall(r'\d+', soup.find('p', class_='instock availability').text)[0]
        book['amount'] = int(amount) if amount else 0
        logging.debug(f"Parsed amount available: {book['amount']}")
    else:
        logging.debug("Amount not found on page.")

    # Рейтинг
    # Значение рейтинга хранится в классе после класса 'star-rating'
    rating = soup.find("p", attrs={'class': 'star-rating'}).get("class")[1]
    if rating:
        # Получаем числовое значение рейтинга
        book['rating'] = {
            "One": 1,
            "Two": 2,
            "Three": 3,
            "Four": 4,
            "Five": 5
        }.get(rating)
        logging.debug(f"Parsed rating: {book['rating']}")
    else:
        logging.debug("Rating not found on page.")

    # Описание
    description = soup.find('meta', {'name': 'description'})
    if description:
        book['description'] = description['content']
        logging.debug(f"Parsed description: {book['description']}")
    else:
        logging.debug("Description not found on page.")

    # Дополнительные характеристики
    additional_info = soup.find('table', class_='table table-striped')
    info_table = additional_info.find_all('tr')
    if info_table:
        logging.debug("Parsing additional info...")
        for row in info_table:
            book['additional_info'][row.th.text] = row.td.text
            logging.debug(f"{row.th.text}: {row.td.text}")
        logging.debug("Finished parsing additional info...")
    else:
        logging.debug("Additional info not found on page.")
    return book
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [3]:
# Используйте для самопроверки
book_url = 'http://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html'
get_book_data(book_url)

{'title': 'A Light in the Attic',
 'price': '£51.77',
 'rating': 3,
 'amount': 22,
 'description': "\n    It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love that Silverstein. Need proof of his genius? RockabyeRockabye baby, in the treetopDon't you know a treetopIs no safe place to rock?And who put you

## Задание 2. Сбор данных обо всех книгах (20 баллов)

Создайте функцию `scrape_books`, которая будет проходиться по всем страницам из каталога (вида `http://books.toscrape.com/catalogue/page-{N}.html`) и осуществлять парсинг всех страниц в цикле, используя ранее написанную `get_book_data`.

Добавьте аргумент-флаг, который будет отвечать за сохранение результата в файл: если он будет равен `True`, то информация сохранится в ту же папку в файл `books_data.txt`; иначе шаг сохранения будет пропущен.

**Также не забывайте про соблюдение PEP-8**

In [4]:
# Ссылка на каталог, где %d - номер страницы
catalog_url = 'https://books.toscrape.com/catalogue/page-%d.html'
# Ссылка на страницу с книгой, где %s - relative URL книги
book_url = 'https://books.toscrape.com/catalogue/%s'


def books_url_producer(queue: Queue, pages_count: int):
    """
        Producer: собирает с каждой страницы каталога URL книг
        и добавляет в очередь для Consumer для последующего парсинга

        queue (Queue): Очередь, которая заполняется URL адресами книг
        pages_count (int): Количество страниц в каталоге
    """
    logging.debug('URL producer started...')
    # Получаем информацию о книгах с каждой страницы каталога
    for i in range(1, pages_count + 1):
        logging.debug(f'Catalog: Page №{i}')
        soup = BeautifulSoup(requests.get(
                    catalog_url % i, timeout=10).content,
                    'html.parser')
        books = soup.find_all('article', class_='product_pod')
        # Формируем ссылки по каждой книге и добавляем в очередь
        for book in books:
            logging.debug(f"Got book ref source: {book.a['href']}")
            queue.put(book_url % book.a['href'])


def book_data_consumer(queue: Queue, result_list: list):
    """
    Consumer: берет URL из очереди и возвращает информацию о книге
    через get_book_data

    Args:
        queue (Queue): Очередь URL адресов книг
        result_list (list): Список для записи результата парсинга книг
    """
    logging.debug('Book consumer started...')
    try:
        while True:
            # Парсим книги по ссылкам из очереди пока она не пуста (ожидаем до 5 секунд)
            url = queue.get(timeout=5)
            if url is None:
                break
            book_data = get_book_data(url)
            result_list.append(book_data)
            logging.debug(f'Consumer processed: {url}')

    except Empty:
        logging.debug('Consumer: queue empty, finishing...')
    except Exception as e:
        logging.error(f"Consumer error: {e}")


def scrape_books(is_save=False) -> list:
    """
    Посылает GET запрос по URL адресу страницы с каталогом.
    Получает информацию о количестве страниц в каталоге.
    Запускает books_url_producer для сохранения ссылок на книги в очередь
    и book_data_consumer для парсинга книг через очередь из ссылок

    Сохраняет результат парсинга в файл books_data.txt

    Args:
        is_save (bool): Флаг сохранения результата в файл books_data.txt
    Returns:
        result (list): Список информации о книгах на сайте:
        название, цена, рейтинг, количество в наличии, описание и
        дополнительные характеристики
    """
    result = []
    # НАЧАЛО ВАШЕГО РЕШЕНИЯ
    logging.info('Scraping books from books.toscrape.com ...')

    # Получаем информацию о количестве страниц в каталоге через пейджер
    soup = BeautifulSoup(requests.get(catalog_url % 1).content, 'html.parser')
    pages_count = int(re.findall(r'Page 1 of\s+(\d+)', soup.find('ul', class_='pager').li.text)[0])
    books_url_queue = Queue()

    with ThreadPoolExecutor(max_workers=11) as executor:
        producer_future = executor.submit(books_url_producer, books_url_queue, pages_count)
        consumer_futures = [
            executor.submit(book_data_consumer, books_url_queue, result)
            for _ in range(10)
        ]
        producer_future.result()
        logging.info('Producer finished')
        for _ in range(10):
            books_url_queue.put(None)
        wait(consumer_futures)

    logging.info('Scraping finished')
    if is_save:
        # Сохраняем результат в books_data.txt
        with open('books_data.txt', 'w', encoding='utf-8') as file:
            json.dump(result, file, ensure_ascii=False, indent=4)
            logging.info('Data successfully saved to books_data.txt')
    return result
    # КОНЕЦ ВАШЕГО РЕШЕНИЯ

In [5]:
# Проверка работоспособности функции
res = scrape_books(is_save=True)  # Допишите ваши аргументы
print(type(res), len(res))  # и проверки

<class 'list'> 1000


## Задание 3. Настройка регулярной выгрузки (10 баллов)

Настройте автоматический запуск функции сбора данных каждый день в 19:00.
Для автоматизации используйте библиотеку `schedule`. Функция должна запускаться в указанное время и сохранять обновленные данные в текстовый файл.



Бесконечный цикл должен обеспечивать постоянное ожидание времени для запуска задачи и выполнять ее по расписанию. Однако чтобы не перегружать систему, стоит подумать о том, чтобы выполнять проверку нужного времени не постоянно, а раз в какой-то промежуток. В этом вам может помочь `time.sleep(...)`.

Проверьте работоспособность кода локально на любом времени чч:мм.



In [6]:
# НАЧАЛО ВАШЕГО РЕШЕНИЯ

# Расписание
SCHEDULE_TIME = '14:52'

# Сброс предыдущих задач
schedule.clear()

# Для удобства отладки и демонастрации результата добавлено логгирование
# Сброс старых хендлеров логгирования
for h in logging.getLogger().handlers[:]:
    logging.getLogger().removeHandler(h)

# Настройка логгирования
logging.basicConfig(
    level=logging.DEBUG,
    filename='../scrape_books.log',
    filemode='a',
    encoding='utf-8',
    format='%(asctime)s [%(levelname)s] %(message)s',
    force=True)

# Вывод логгирования в консоль
console = logging.StreamHandler()
console.setLevel(logging.INFO)
console.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))
logging.getLogger().addHandler(console)

# Расписание запуска джобы вынесено в SCHEDULE_TIME
job = schedule.every().day.at(SCHEDULE_TIME).do(scrape_books, is_save=True)
logging.info(f"Scheduler initialized. Job will run daily at {SCHEDULE_TIME}.")

try:
    logging.info(f"Next run scheduled for {job.next_run.strftime('%Y-%m-%d %H:%M')}")
except Exception:
    logging.debug("Job registered")

while True:
    schedule.run_pending()
    time.sleep(60)
# КОНЕЦ ВАШЕГО РЕШЕНИЯ

2025-11-08 14:51:44,414 [INFO] Scheduler initialized. Job will run daily at 14:52.
2025-11-08 14:51:44,420 [INFO] Next run scheduled for 2025-11-08 14:52:00
2025-11-08 14:52:44,421 [INFO] Scraping books from books.toscrape.com ...
2025-11-08 14:53:52,319 [INFO] Producer finished
2025-11-08 14:54:52,300 [INFO] Scraping finished
2025-11-08 14:54:52,337 [INFO] Data successfully saved to books_data.txt


KeyboardInterrupt: 

## Задание 4. Написание автотестов (15 баллов)

Создайте минимум три автотеста для ключевых функций парсинга — например, `get_book_data` и `scrape_books`. Идеи проверок (можете использовать свои):

* данные о книге возвращаются в виде словаря с нужными ключами;
* список ссылок или количество собранных книг соответствует ожиданиям;
* значения отдельных полей (например, `title`) корректны.

Оформите тесты в отдельном скрипте `tests/test_scraper.py`, используйте библиотеку `pytest`. Убедитесь, что тесты проходят успешно при запуске из терминала командой `pytest`.

Также выведите результат их выполнения в ячейке ниже.

**Не забывайте про соблюдение PEP-8**


In [7]:
# Ячейка для демонстрации работоспособности
# Сам код напишите в отдельном скрипте
! pytest ../tests/test_scraper.py

============================= test session starts =============================
platform win32 -- Python 3.13.7, pytest-8.4.2, pluggy-1.6.0
rootdir: C:\Users\Alisa\PycharmProjects\book-scraper
configfile: pytest.ini
plugins: anyio-4.11.0
collected 5 items

..\tests\test_scraper.py .....                                           [100%]

======================== 5 passed in 139.20s (0:02:19) ========================


## Задание 5. Оформление проекта на GitHub и работа с Git (35 баллов)

В этом задании нужно воспользоваться системой контроля версий Git и платформой GitHub для хранения и управления своим проектом. **Ссылку на свой репозиторий пришлите в форме для сдачи ответа.**

### Пошаговая инструкция и задания

**1. Установите Git на свой компьютер.**

* Для Windows: [скачайте установщик](https://git-scm.com/downloads) и выполните установку.
* Для macOS:

  ```
  brew install git
  ```
* Для Linux:

  ```
  sudo apt update
  sudo apt install git
  ```

**2. Настройте имя пользователя и email.**

Это нужно для подписи ваших коммитов, сделайте в терминале через `git config ...`.

**3. Создайте аккаунт на GitHub**, если у вас его еще нет:
[https://github.com](https://github.com)

**4. Создайте новый репозиторий на GitHub:**

* Найдите кнопку **New repository**.
* Укажите название, краткое описание, выберите тип **Public** (чтобы мы могли проверить ДЗ).
* Не ставьте галочку Initialize this repository with a README.

**5. Создайте локальную папку с проектом.** Можно в терминале, можно через UI, это не имеет значения.

**6. Инициализируйте Git в этой папке.** Здесь уже придется воспользоваться некоторой командой в терминале.

**7. Привяжите локальный репозиторий к удаленному на GitHub.**

**8. Создайте ветку разработки.** По умолчанию вы будете находиться в ветке `main`, создайте и переключитесь на ветку `hw-books-parser`.

**9. Добавьте в проект следующие файлы и папки:**

* `scraper.py` — ваш основной скрипт для сбора данных.
* `README.md` — файл с кратким описанием проекта:

  * цель;
  * инструкции по запуску;
  * список используемых библиотек.
* `requirements.txt` — файл со списком зависимостей, необходимых для проекта (не присылайте все из глобального окружения, создайте изолированную виртуальную среду, добавьте в нее все нужное для проекта и получите список библиотек через `pip freeze`).
* `artifacts/` — папка с результатами парсинга (`books_data.txt` — полностью или его часть, если весь не поместится на GitHub).
* `notebooks/` — папка с заполненным ноутбуком `HW_03_python_ds_2025.ipynb` и запущенными ячейками с выводами на экран.
* `tests/` — папка с тестами на `pytest`, оформите их в формате скрипта(-ов) с расширением `.py`.
* `.gitignore` — стандартный файл, который позволит исключить временные файлы при добавлении в отслеживаемые (например, `__pycache__/`, `.DS_Store`, `*.pyc`, `venv/` и др.).


**10. Сделайте коммит.**

**11. Отправьте свою ветку на GitHub.**

**12. Создайте Pull Request:**

* Перейдите в репозиторий на GitHub.
* Нажмите кнопку **Compare & pull request**.
* Укажите, что было добавлено, и нажмите **Create pull request**.

**13. Выполните слияние Pull Request:**

* Убедитесь, что нет конфликтов.
* Нажмите **Merge pull request**, затем **Confirm merge**.

**14. Скачайте изменения из основной ветки локально.**



### Требования к итоговому репозиторию

* Файл `scraper.py` с рабочим кодом парсера.
* `README.md` с описанием проекта и инструкцией по запуску.
* Папка `artifacts/` с результатом сбора данных (`.txt` файл).
* Папка `tests/` с тестами на `pytest`.
* Папка `notebooks/` с заполненным ноутбуком `HW_03_python_ds_2025.ipynb`.
* Pull Request с комментарием из ветки `hw-books-parser` в ветку `main`.
* Примерная структура:

  ```
  books_scraper/
  ├── artifacts/
  │   └── books_data.txt
  ├── notebooks/
  │   └── HW_03_python_ds_2025.ipynb
  ├── scraper.py
  ├── README.md
  ├── tests/
  │   └── test_scraper.py
  ├── .gitignore
  └── requirements.txt
  ```